# PRISM-U Stage 2 — Complete Colab Runner

This notebook drives the complete Stage 2 implementation in the repository. It does not duplicate the research code inside notebook cells; the repository remains the source of truth.

**Before running:** use Runtime → Change runtime type → GPU. Because the repository is private, clone/authenticate it in Colab using your own GitHub method. Do not paste a personal access token into a saved notebook.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
REPO = Path('/content/PRISM-U')
assert REPO.is_dir(), ('Clone hamza26051/PRISM-U into /content/PRISM-U first. '
                       'The notebook intentionally does not store GitHub credentials.')
print('Repository found:', REPO)

In [ ]:
import subprocess
subprocess.run(['git', '-C', str(REPO), 'checkout', 'stage2-complete'], check=True)
subprocess.run(['git', '-C', str(REPO), 'lfs', 'install'], check=True)
subprocess.run(['git', '-C', str(REPO), 'lfs', 'pull'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(REPO / 'stage2/requirements.txt')], check=True)

## Persist Stage 2 artifacts in Drive
The trained bank is several GB and must not live only in Colab's temporary `/content`. The next cell changes only the artifact output path in the Stage 2 config.

In [ ]:
import yaml
CONFIG = REPO / 'stage2/configs/stage2.yaml'
cfg = yaml.safe_load(CONFIG.read_text())
cfg['paths']['artifact_root'] = '/content/drive/MyDrive/PRISM-U-stage2-artifacts'
CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
print('Artifact root:', cfg['paths']['artifact_root'])

In [ ]:
%cd /content/PRISM-U
!python stage2/scripts/verify_inputs.py
!python -m pytest stage2/tests -q

## Train one family per run/session
Run this cell repeatedly, changing `ROLE` and `SEED`. Complete surrogate 101–112, validation 201–204, then final 42. Training is resumable.

In [ ]:
ROLE = 'surrogate'   # surrogate | validation | final
SEED = 101            # surrogate: 101..112; validation: 201..204; final: 42
!python stage2/run_stage2.py --phase bank --role {ROLE} --seed {SEED}


## After ALL 17 families exist
Only run the following cell after surrogate seeds 101–112, validation seeds 201–204, and final seed 42 have all completed. It freezes the 5,000-query pool, caches responses, calculates diagnostics, and runs the completion gate.

In [ ]:
!python stage2/run_stage2.py --phase responses
!python stage2/run_stage2.py --phase diagnostics
!python stage2/run_stage2.py --phase validate

In [ ]:
import json
report = Path('/content/drive/MyDrive/PRISM-U-stage2-artifacts/validation_report.json')
data = json.loads(report.read_text())
print(json.dumps(data, indent=2))
assert data['stage2_complete'] is True
print('\nPRISM-U STAGE 2 IS COMPLETE.')